# 用向量嵌入、OpenAI 和 Cassandra / Astra DB 探索哲学

### CQL 版本

在本快速入门中，你将学习如何使用 OpenAI 的向量嵌入，并以 [Apache Cassandra®](https://cassandra.apache.org)（或等价的 DataStax [通过 CQL 访问的 Astra DB](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html)）作为持久化数据的向量存储，构建一个“哲学名言查找与生成器”。

本 notebook 的基本工作流程概述如下。你将为一批著名哲学家的名言计算并存储向量嵌入，用它们构建一个强大的搜索引擎，之后甚至还能构建一个新名言的生成器！

本 notebook 示范了向量搜索的一些标准使用模式——同时展示了上手 [Cassandra](https://cassandra.apache.org/doc/trunk/cassandra/vector-search/overview.html) / [通过 CQL 访问的 Astra DB](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html) 的向量能力是多么容易。

关于使用向量搜索和文本嵌入构建问答系统的背景知识，请参阅这篇出色的实践 notebook：[使用嵌入进行问答](https://github.com/openai/openai-cookbook/blob/main/examples/Question_answering_using_embeddings.ipynb)。

#### _选择你的框架_

请注意，本 notebook 使用 [Cassandra 驱动](https://docs.datastax.com/en/developer/python-driver/latest/) 并直接运行 CQL（Cassandra Query Language）语句，但我们也介绍了完成同一任务的其他技术选择。其他选项请查看此文件夹的 [README](https://github.com/openai/openai-cookbook/tree/main/examples/vector_databases/cassandra_astradb)。本 notebook 既可以作为 Colab notebook 运行，也可以作为普通的 Jupyter notebook 运行。

目录：
- 环境设置
- 获取数据库连接
- 连接 OpenAI
- 将名言加载到向量存储中
- 用例 1：**名言搜索引擎**
- 用例 2：**名言生成器**
- （可选）在向量存储中利用分区

### 工作原理

**索引**

每条名言都通过 OpenAI 的 `Embedding` 转换为嵌入向量。这些向量保存在向量存储中，供之后搜索使用。一些元数据（包括作者姓名和其他几个预先计算的标签）也会一并存储，以便定制搜索。

![1_vector_indexing_cql](https://user-images.githubusercontent.com/14221764/282437237-1e763166-a863-4332-99b8-323ba23d1b87.png)

**搜索**

为了找到与给定搜索名言相似的名言，会实时将后者转换为嵌入向量，并用这个向量在存储中查询相似的向量……也就是之前已建立索引的相似名言。搜索还可以选择性地通过额外的元数据进行限制（“帮我找出斯宾诺莎的、与这句相似的名言……”）。

![2_vector_search_cql](https://user-images.githubusercontent.com/14221764/282437291-85335612-a845-444e-bed7-e4cf014a9f17.png)

这里的关键在于，“内容相似的名言”在向量空间中对应于彼此在度量上相近的向量：因此，向量相似度搜索实际上实现了语义相似度。_这正是向量嵌入如此强大的关键原因。_

下面的示意图试图传达这一思想。每条名言一旦转换为向量，就是空间中的一个点。嗯，在本例中它位于一个球面上，因为 OpenAI 的嵌入向量和大多数其他嵌入向量一样，都被归一化为_单位长度_。哦，而且这个球面其实不是三维的，而是 1536 维的！

因此，从本质上讲，向量空间中的相似度搜索返回的是与查询向量最接近的那些向量：

![3_vector_space](https://user-images.githubusercontent.com/14221764/262321363-c8c625c1-8be9-450e-8c68-b1ed518f990d.png)

**生成**

给定一个建议（一个主题或一句暂定的名言），先执行搜索步骤，然后将返回的前几个结果（名言）填入 LLM 提示词，要求生成模型按照所给示例_以及_最初的建议创作一段新文本。

![4_quote_generation](https://user-images.githubusercontent.com/14221764/282437321-881bd273-3443-4987-9a11-350d3288dd8e.png)

## 环境设置

安装并导入必要的依赖：

In [1]:
!pip install --quiet "cassandra-driver>=0.28.0" "openai>=1.0.0" datasets

In [2]:
import os
from uuid import uuid4
from getpass import getpass
from collections import Counter

from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

import openai
from datasets import load_dataset

_不必太在意下一个单元格，我们需要它来检测 Colab 环境，并让你上传 SCB 文件（见下文）：_

In [3]:
try:
    from google.colab import files
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

## 获取数据库连接

创建 `Session` 对象（即与你的 Astra DB 实例的连接）需要几个密钥。

_（注意：某些步骤在 Google Colab 和本地 Jupyter 上会略有不同，因此 notebook 会检测运行时类型。）_

In [4]:
# Your database's Secure Connect Bundle zip file is needed:
if IS_COLAB:
    print('Please upload your Secure Connect Bundle zipfile: ')
    uploaded = files.upload()
    if uploaded:
        astraBundleFileTitle = list(uploaded.keys())[0]
        ASTRA_DB_SECURE_BUNDLE_PATH = os.path.join(os.getcwd(), astraBundleFileTitle)
    else:
        raise ValueError(
            'Cannot proceed without Secure Connect Bundle. Please re-run the cell.'
        )
else:
    # you are running a local-jupyter notebook:
    ASTRA_DB_SECURE_BUNDLE_PATH = input("Please provide the full path to your Secure Connect Bundle zipfile: ")

ASTRA_DB_APPLICATION_TOKEN = getpass("Please provide your Database Token ('AstraCS:...' string): ")
ASTRA_DB_KEYSPACE = input("Please provide the Keyspace name for your Database: ")

Please provide the full path to your Secure Connect Bundle zipfile:  /path/to/secure-connect-DatabaseName.zip
Please provide your Database Token ('AstraCS:...' string):  ········
Please provide the Keyspace name for your Database:  my_keyspace


### 创建数据库连接

以下是创建 Astra DB 连接的方法：

_（顺便一提，你也可以使用任何 Cassandra 集群（只要它提供向量能力），只需对下面的 `Cluster` 实例化[修改参数](https://docs.datastax.com/en/developer/python-driver/latest/getting_started/#connecting-to-cassandra)即可。）_

In [5]:
# Don't mind the "Closing connection" error after "downgrading protocol..." messages you may see,
# it is really just a warning: the connection will work smoothly.
cluster = Cluster(
    cloud={
        "secure_connect_bundle": ASTRA_DB_SECURE_BUNDLE_PATH,
    },
    auth_provider=PlainTextAuthProvider(
        "token",
        ASTRA_DB_APPLICATION_TOKEN,
    ),
)

session = cluster.connect()
keyspace = ASTRA_DB_KEYSPACE

### 在 CQL 中创建向量表

你需要一张支持向量并带有元数据的表。将其命名为“philosophers_cql”。

每一行将存储：一条名言、它的向量嵌入、名言作者以及一组“标签”。你还需要一个主键来确保行的唯一性。

下面是创建该表的完整 CQL 命令（关于这条语句及后续语句的 CQL 语法，请参阅[此页面](https://docs.datastax.com/en/dse/6.7/cql/cql/cqlQuickReference.html)）：

In [6]:
create_table_statement = f"""CREATE TABLE IF NOT EXISTS {keyspace}.philosophers_cql (
    quote_id UUID PRIMARY KEY,
    body TEXT,
    embedding_vector VECTOR<FLOAT, 1536>,
    author TEXT,
    tags SET<TEXT>
);"""

将这条语句传给数据库 Session 执行：

In [7]:
session.execute(create_table_statement)

#### 为 ANN 搜索添加向量索引

为了对表中的向量执行 ANN（近似最近邻）搜索，你需要在 `embedding_vector` 列上创建一个专门的索引。

_创建索引时，你可以[选择性地指定](https://docs.datastax.com/en/astra-serverless/docs/vector-search/cql.html#_create_the_vector_schema_and_load_the_data_into_the_database)用于计算向量距离的“相似度函数”：由于对于单位长度的向量（例如 OpenAI 的向量）而言，“余弦差异”与“点积”是相同的，因此你将使用计算开销更小的后者。_

运行以下 CQL 语句：

In [8]:
create_vector_index_statement = f"""CREATE CUSTOM INDEX IF NOT EXISTS idx_embedding_vector
    ON {keyspace}.philosophers_cql (embedding_vector)
    USING 'org.apache.cassandra.index.sai.StorageAttachedIndex'
    WITH OPTIONS = {{'similarity_function' : 'dot_product'}};
"""
# Note: the double '{{' and '}}' are just the F-string escape sequence for '{' and '}'

session.execute(create_vector_index_statement)

#### 为作者和标签过滤添加索引

这已足以在表上运行向量搜索了……但你还希望能够选择性地指定作者和/或某些标签来限制名言搜索。再创建两个索引来支持这一点：

In [9]:
create_author_index_statement = f"""CREATE CUSTOM INDEX IF NOT EXISTS idx_author
    ON {keyspace}.philosophers_cql (author)
    USING 'org.apache.cassandra.index.sai.StorageAttachedIndex';
"""
session.execute(create_author_index_statement)

create_tags_index_statement = f"""CREATE CUSTOM INDEX IF NOT EXISTS idx_tags
    ON {keyspace}.philosophers_cql (VALUES(tags))
    USING 'org.apache.cassandra.index.sai.StorageAttachedIndex';
"""
session.execute(create_tags_index_statement)

## 连接 OpenAI

### 设置你的密钥

In [10]:
OPENAI_API_KEY = getpass("Please enter your OpenAI API Key: ")

Please enter your OpenAI API Key:  ········


### 测试嵌入调用

快速看一下如何为一组输入文本获取嵌入向量：

In [11]:
client = openai.OpenAI(api_key=OPENAI_API_KEY)
embedding_model_name = "text-embedding-3-small"

result = client.embeddings.create(
    input=[
        "This is a sentence",
        "A second sentence"
    ],
    model=embedding_model_name,
)

_注意：以上是 OpenAI v1.0+ 的语法。如果使用之前的版本，获取嵌入的代码会有所不同。_

In [12]:
print(f"len(result.data)              = {len(result.data)}")
print(f"result.data[1].embedding      = {str(result.data[1].embedding)[:55]}...")
print(f"len(result.data[1].embedding) = {len(result.data[1].embedding)}")

len(result.data)              = 2
result.data[1].embedding      = [-0.0108176339417696, 0.0013546717818826437, 0.00362232...
len(result.data[1].embedding) = 1536


## 将名言加载到向量存储中

获取包含名言的数据集。_（我们对[这个 Kaggle 数据集](https://www.kaggle.com/datasets/mertbozkurt5/quotes-by-philosophers)中的数据进行了改编和扩充，可直接用于本演示。）_

In [13]:
philo_dataset = load_dataset("datastax/philosopher-quotes")["train"]

快速查看一下：

In [14]:
print("An example entry:")
print(philo_dataset[16])

An example entry:
{'author': 'aristotle', 'quote': 'Love well, be loved and do something of value.', 'tags': 'love;ethics'}


检查数据集大小：

In [15]:
author_count = Counter(entry["author"] for entry in philo_dataset)
print(f"Total: {len(philo_dataset)} quotes. By author:")
for author, count in author_count.most_common():
    print(f"    {author:<20}: {count} quotes")

Total: 450 quotes. By author:
    aristotle           : 50 quotes
    schopenhauer        : 50 quotes
    spinoza             : 50 quotes
    hegel               : 50 quotes
    freud               : 50 quotes
    nietzsche           : 50 quotes
    sartre              : 50 quotes
    plato               : 50 quotes
    kant                : 50 quotes


### 将名言插入向量存储

你将为名言计算嵌入，并将其连同文本本身以及计划后续使用的元数据一起保存到向量存储中。

为了优化速度并减少调用次数，你将对 OpenAI 嵌入服务进行批量调用。

数据库写入通过一条 CQL 语句完成。但由于你会多次运行这个插入操作（尽管值各不相同），最好先_预处理（prepare）_该语句，然后反复执行它。

_（注意：为了加快插入速度，Cassandra 驱动支持并发插入，但为了让演示代码更直观，这里没有这样做。）_

In [16]:
prepared_insertion = session.prepare(
    f"INSERT INTO {keyspace}.philosophers_cql (quote_id, author, body, embedding_vector, tags) VALUES (?, ?, ?, ?, ?);"
)

BATCH_SIZE = 20

num_batches = ((len(philo_dataset) + BATCH_SIZE - 1) // BATCH_SIZE)

quotes_list = philo_dataset["quote"]
authors_list = philo_dataset["author"]
tags_list = philo_dataset["tags"]

print("Starting to store entries:")
for batch_i in range(num_batches):
    b_start = batch_i * BATCH_SIZE
    b_end = (batch_i + 1) * BATCH_SIZE
    # compute the embedding vectors for this batch
    b_emb_results = client.embeddings.create(
        input=quotes_list[b_start : b_end],
        model=embedding_model_name,
    )
    # prepare the rows for insertion
    print("B ", end="")
    for entry_idx, emb_result in zip(range(b_start, b_end), b_emb_results.data):
        if tags_list[entry_idx]:
            tags = {
                tag
                for tag in tags_list[entry_idx].split(";")
            }
        else:
            tags = set()
        author = authors_list[entry_idx]
        quote = quotes_list[entry_idx]
        quote_id = uuid4()  # a new random ID for each quote. In a production app you'll want to have better control...
        session.execute(
            prepared_insertion,
            (quote_id, author, quote, emb_result.embedding, tags),
        )
        print("*", end="")
    print(f" done ({len(b_emb_results.data)})")

print("\nFinished storing entries.")

Starting to store entries:
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ******************** done (20)
B ********** done (10)

Finished storing entries.


## 用例 1：**名言搜索引擎**

对于名言搜索功能，你首先需要将输入的名言转换为向量，然后用它查询存储（此外还要在搜索调用中处理可选的元数据）。

将搜索引擎功能封装到一个函数中，以便复用：

In [17]:
def find_quote_and_author(query_quote, n, author=None, tags=None):
    query_vector = client.embeddings.create(
        input=[query_quote],
        model=embedding_model_name,
    ).data[0].embedding
    # depending on what conditions are passed, the WHERE clause in the statement may vary.
    where_clauses = []
    where_values = []
    if author:
        where_clauses += ["author = %s"]
        where_values += [author]
    if tags:
        for tag in tags:
            where_clauses += ["tags CONTAINS %s"]
            where_values += [tag]
    # The reason for these two lists above is that when running the CQL search statement the values passed
    # must match the sequence of "?" marks in the statement.
    if where_clauses:
        search_statement = f"""SELECT body, author FROM {keyspace}.philosophers_cql
            WHERE {' AND '.join(where_clauses)}
            ORDER BY embedding_vector ANN OF %s
            LIMIT %s;
        """
    else:
        search_statement = f"""SELECT body, author FROM {keyspace}.philosophers_cql
            ORDER BY embedding_vector ANN OF %s
            LIMIT %s;
        """
    # For best performance, one should keep a cache of prepared statements (see the insertion code above)
    # for the various possible statements used here.
    # (We'll leave it as an exercise to the reader to avoid making this code too long.
    # Remember: to prepare a statement you use '?' instead of '%s'.)
    query_values = tuple(where_values + [query_vector] + [n])
    result_rows = session.execute(search_statement, query_values)
    return [
        (result_row.body, result_row.author)
        for result_row in result_rows
    ]

### 测试搜索

仅传入一条名言：

In [18]:
find_quote_and_author("We struggle all our life for nothing", 3)

[('Life to the great majority is only a constant struggle for mere existence, with the certainty of losing it at last.',
  'schopenhauer'),
 ('We give up leisure in order that we may have leisure, just as we go to war in order that we may have peace.',
  'aristotle'),
 ('Perhaps the gods are kind to us, by making life more disagreeable as we grow older. In the end death seems less intolerable than the manifold burdens we carry',
  'freud')]

限定作者的搜索：

In [19]:
find_quote_and_author("We struggle all our life for nothing", 2, author="nietzsche")

[('To live is to suffer, to survive is to find some meaning in the suffering.',
  'nietzsche'),
 ('What makes us heroic?--Confronting simultaneously our supreme suffering and our supreme hope.',
  'nietzsche')]

限定标签的搜索（从之前与名言一起保存的标签中选择）：

In [20]:
find_quote_and_author("We struggle all our life for nothing", 2, tags=["politics"])

[('Mankind will never see an end of trouble until lovers of wisdom come to hold political power, or the holders of power become lovers of wisdom',
  'plato'),
 ('Everything the State says is a lie, and everything it has it has stolen.',
  'nietzsche')]

### 剔除不相关的结果

向量相似度搜索通常会返回与查询最接近的向量，即使在没有更好结果的情况下，这些结果可能有些不相关。

为了控制这个问题，你可以获取查询与每个结果之间实际的“相似度”，然后为其设定一个截断值，从而丢弃超出该阈值的结果。
正确调整这个阈值并不是一个简单的问题：这里我们只演示具体做法。

为了直观感受其效果，请尝试以下查询，并调整名言和阈值的选择来比较结果：

_注（写给喜欢数学的读者）：这个值将余弦相似度从 -1 到 1 的范围重新缩放到 0 到 1 的范围。余弦相似度是标量积除以两个向量范数的乘积。对于方向相反的向量，缩放后的值为 0；对于平行的向量，值为 1。其他相似度度量请查阅[文档](https://docs.datastax.com/en/astra-serverless/docs/vector-search/cql.html#_create_the_vector_schema_and_load_the_data_into_the_database)，并请记住，`SELECT` 查询中的度量应与创建索引时使用的度量一致，才能得到有意义且有序的结果。_

In [21]:
quote = "Animals are our equals."
# quote = "Be good."
# quote = "This teapot is strange."

similarity_threshold = 0.92

quote_vector = client.embeddings.create(
    input=[quote],
    model=embedding_model_name,
).data[0].embedding

# Once more: remember to prepare your statements in production for greater performance...

search_statement = f"""SELECT body, similarity_dot_product(embedding_vector, %s) as similarity
    FROM {keyspace}.philosophers_cql
    ORDER BY embedding_vector ANN OF %s
    LIMIT %s;
"""
query_values = (quote_vector, quote_vector, 8)

result_rows = session.execute(search_statement, query_values)
results = [
    (result_row.body, result_row.similarity)
    for result_row in result_rows
    if result_row.similarity >= similarity_threshold
]

print(f"{len(results)} quotes within the threshold:")
for idx, (r_body, r_similarity) in enumerate(results):
    print(f"    {idx}. [similarity={r_similarity:.3f}] \"{r_body[:70]}...\"")

3 quotes within the threshold:
    0. [similarity=0.927] "The assumption that animals are without rights, and the illusion that ..."
    1. [similarity=0.922] "Animals are in possession of themselves; their soul is in possession o..."
    2. [similarity=0.920] "At his best, man is the noblest of all animals; separated from law and..."


## 用例 2：**名言生成器**

完成这项任务需要 OpenAI 的另一个组件，即一个为我们生成名言的 LLM（基于查询向量存储得到的输入）。

你还需要一个提示词模板，用于填充生成名言的 LLM 补全任务。

In [22]:
completion_model_name = "gpt-3.5-turbo"

generation_prompt_template = """"Generate a single short philosophical quote on the given topic,
similar in spirit and form to the provided actual example quotes.
Do not exceed 20-30 words in your quote.

REFERENCE TOPIC: "{topic}"

ACTUAL EXAMPLES:
{examples}
"""

与搜索一样，这个功能最好封装成一个便捷函数（其内部会使用搜索）：

In [23]:
def generate_quote(topic, n=2, author=None, tags=None):
    quotes = find_quote_and_author(query_quote=topic, n=n, author=author, tags=tags)
    if quotes:
        prompt = generation_prompt_template.format(
            topic=topic,
            examples="\n".join(f"  - {quote[0]}" for quote in quotes),
        )
        # a little logging:
        print("** quotes found:")
        for q, a in quotes:
            print(f"**    - {q} ({a})")
        print("** end of logging")
        #
        response = client.chat.completions.create(
            model=completion_model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=320,
        )
        return response.choices[0].message.content.replace('"', '').strip()
    else:
        print("** no quotes found.")
        return None

_注意：与嵌入计算的情况类似，在 OpenAI v1.0 之前的版本中，Chat Completion API 的代码会略有不同。_

#### 测试名言生成

仅传入一段文本（一条“名言”，但实际上也可以只给出一个主题，因为它的向量嵌入仍会落在向量空间中的正确位置）：

In [24]:
q_topic = generate_quote("politics and virtue")
print("\nA new generated quote:")
print(q_topic)

** quotes found:
**    - Happiness is the reward of virtue. (aristotle)
**    - Our moral virtues benefit mainly other people; intellectual virtues, on the other hand, benefit primarily ourselves; therefore the former make us universally popular, the latter unpopular. (schopenhauer)
** end of logging

A new generated quote:
True politics is not the pursuit of power, but the cultivation of virtue for the betterment of all.


仅从一位哲学家那里汲取灵感：

In [25]:
q_topic = generate_quote("animals", author="schopenhauer")
print("\nA new generated quote:")
print(q_topic)

** quotes found:
**    - Because Christian morality leaves animals out of account, they are at once outlawed in philosophical morals; they are mere 'things,' mere means to any ends whatsoever. They can therefore be used for vivisection, hunting, coursing, bullfights, and horse racing, and can be whipped to death as they struggle along with heavy carts of stone. Shame on such a morality that is worthy of pariahs, and that fails to recognize the eternal essence that exists in every living thing, and shines forth with inscrutable significance from all eyes that see the sun! (schopenhauer)
**    - The assumption that animals are without rights, and the illusion that our treatment of them has no moral significance, is a positively outrageous example of Western crudity and barbarity. Universal compassion is the only guarantee of morality. (schopenhauer)
** end of logging

A new generated quote:
Do not judge the worth of a soul by its outward form, for within every animal lies an eternal esse

## （可选）**分区**

在结束本快速入门之前，还有一个有趣的话题值得探讨。一般来说，标签与名言之间可以是任意关系（例如一条名言有多个标签），但_作者_实际上构成了一种精确的分组（它们在名言集合上定义了一种“不相交的划分”）：每条名言恰好有一位作者（至少对我们而言如此）。

现在，假设你事先知道你的应用通常（或总是）只针对_单个作者_运行查询。那么你就可以充分利用底层数据库结构：如果将名言按**分区**分组（每位作者一个分区），那么仅针对某位作者的向量查询会占用更少资源，返回速度也快得多。

这里我们不深入细节，这些细节与 Cassandra 存储的内部机制有关：重要的结论是，**如果你的查询都在某个分组内运行，请考虑相应地进行分区以提升性能**。

现在你将看到这一选择的实际效果。

按作者分区需要新的表 schema：创建一张名为“philosophers_cql_partitioned”的新表，以及必要的索引：

In [26]:
create_table_p_statement = f"""CREATE TABLE IF NOT EXISTS {keyspace}.philosophers_cql_partitioned (
    author TEXT,
    quote_id UUID,
    body TEXT,
    embedding_vector VECTOR<FLOAT, 1536>,
    tags SET<TEXT>,
    PRIMARY KEY ( (author), quote_id )
) WITH CLUSTERING ORDER BY (quote_id ASC);"""

session.execute(create_table_p_statement)

create_vector_index_p_statement = f"""CREATE CUSTOM INDEX IF NOT EXISTS idx_embedding_vector_p
    ON {keyspace}.philosophers_cql_partitioned (embedding_vector)
    USING 'org.apache.cassandra.index.sai.StorageAttachedIndex'
    WITH OPTIONS = {{'similarity_function' : 'dot_product'}};
"""

session.execute(create_vector_index_p_statement)

create_tags_index_p_statement = f"""CREATE CUSTOM INDEX IF NOT EXISTS idx_tags_p
    ON {keyspace}.philosophers_cql_partitioned (VALUES(tags))
    USING 'org.apache.cassandra.index.sai.StorageAttachedIndex';
"""
session.execute(create_tags_index_p_statement)

现在在新表上重复“计算嵌入并插入”这一步骤。

你可以使用与之前完全相同的插入代码，因为差异都隐藏在“幕后”：数据库会根据这张新表的分区方案以不同方式存储插入的行。

不过，为了演示，你将利用 Cassandra 驱动提供的一个便捷功能，轻松地并发运行多个查询（本例中是 `INSERT`）。这是 Cassandra / 通过 CQL 访问的 Astra DB 非常擅长支持的功能，只需对客户端代码做很少的修改，就能带来显著的加速。

_（注意：还可以缓存之前计算的嵌入来节省一些 API token——不过这里我们希望让代码更易于查看。）_

In [27]:
from cassandra.concurrent import execute_concurrent_with_args

In [28]:
prepared_insertion = session.prepare(
    f"INSERT INTO {keyspace}.philosophers_cql_partitioned (quote_id, author, body, embedding_vector, tags) VALUES (?, ?, ?, ?, ?);"
)

BATCH_SIZE = 50

num_batches = ((len(philo_dataset) + BATCH_SIZE - 1) // BATCH_SIZE)

quotes_list = philo_dataset["quote"]
authors_list = philo_dataset["author"]
tags_list = philo_dataset["tags"]

print("Starting to store entries:")
for batch_i in range(num_batches):
    print("[...", end="")
    b_start = batch_i * BATCH_SIZE
    b_end = (batch_i + 1) * BATCH_SIZE
    # compute the embedding vectors for this batch
    b_emb_results = client.embeddings.create(
        input=quotes_list[b_start : b_end],
        model=embedding_model_name,
    )
    # prepare this batch's entries for insertion
    tuples_to_insert = []
    for entry_idx, emb_result in zip(range(b_start, b_end), b_emb_results.data):
        if tags_list[entry_idx]:
            tags = {
                tag
                for tag in tags_list[entry_idx].split(";")
            }
        else:
            tags = set()
        author = authors_list[entry_idx]
        quote = quotes_list[entry_idx]
        quote_id = uuid4()  # a new random ID for each quote. In a production app you'll want to have better control...
        # append a *tuple* to the list, and in the tuple the values are ordered to match "?" in the prepared statement:
        tuples_to_insert.append((quote_id, author, quote, emb_result.embedding, tags))
    # insert the batch at once through the driver's concurrent primitive
    conc_results = execute_concurrent_with_args(
        session,
        prepared_insertion,
        tuples_to_insert,
    )
    # check that all insertions succeed (better to always do this):
    if any([not success for success, _ in conc_results]):
        print("Something failed during the insertions!")
    else:
        print(f"{len(b_emb_results.data)}] ", end="")

print("\nFinished storing entries.")

Starting to store entries:
[...50] [...50] [...50] [...50] [...50] [...50] [...50] [...50] [...50] 
Finished storing entries.


尽管表 schema 不同，相似度搜索背后的数据库查询本质上是一样的：

In [29]:
def find_quote_and_author_p(query_quote, n, author=None, tags=None):
    query_vector = client.embeddings.create(
        input=[query_quote],
        model=embedding_model_name,
    ).data[0].embedding
    # Depending on what conditions are passed, the WHERE clause in the statement may vary.
    # Construct it accordingly:
    where_clauses = []
    where_values = []
    if author:
        where_clauses += ["author = %s"]
        where_values += [author]
    if tags:
        for tag in tags:
            where_clauses += ["tags CONTAINS %s"]
            where_values += [tag]
    if where_clauses:
        search_statement = f"""SELECT body, author FROM {keyspace}.philosophers_cql_partitioned
            WHERE {' AND '.join(where_clauses)}
            ORDER BY embedding_vector ANN OF %s
            LIMIT %s;
        """
    else:
        search_statement = f"""SELECT body, author FROM {keyspace}.philosophers_cql_partitioned
            ORDER BY embedding_vector ANN OF %s
            LIMIT %s;
        """
    query_values = tuple(where_values + [query_vector] + [n])
    result_rows = session.execute(search_statement, query_values)
    return [
        (result_row.body, result_row.author)
        for result_row in result_rows
    ]

就是这样：新表依然能很好地支持“通用”相似度搜索……

In [30]:
find_quote_and_author_p("We struggle all our life for nothing", 3)

[('Life to the great majority is only a constant struggle for mere existence, with the certainty of losing it at last.',
  'schopenhauer'),
 ('We give up leisure in order that we may have leisure, just as we go to war in order that we may have peace.',
  'aristotle'),
 ('Perhaps the gods are kind to us, by making life more disagreeable as we grow older. In the end death seems less intolerable than the manifold burdens we carry',
  'freud')]

……但只有在指定作者时，你才会注意到_巨大的_性能优势：

In [31]:
find_quote_and_author_p("We struggle all our life for nothing", 2, author="nietzsche")

[('To live is to suffer, to survive is to find some meaning in the suffering.',
  'nietzsche'),
 ('What makes us heroic?--Confronting simultaneously our supreme suffering and our supreme hope.',
  'nietzsche')]

嗯，如果你有一个真实规模的数据集，你_会_注意到性能提升。在本演示中，只有几十条数据，看不出明显差别——但你应该明白其中的道理了。

## 总结

恭喜！你已经学会了如何使用 OpenAI 生成向量嵌入、使用 Astra DB / Cassandra 进行存储，从而构建一个精巧的哲学搜索引擎和名言生成器。

本示例使用了 [Cassandra 驱动](https://docs.datastax.com/en/developer/python-driver/latest/)，并直接运行 CQL（Cassandra Query Language）语句与向量存储交互——但这并不是唯一的选择。其他选项以及与流行框架的集成，请查看 [README](https://github.com/openai/openai-cookbook/tree/main/examples/vector_databases/cassandra_astradb)。

要进一步了解 Astra DB 的向量搜索能力如何成为你的 ML/GenAI 应用中的关键要素，请访问 [Astra DB](https://docs.datastax.com/en/astra-serverless/docs/vector-search/overview.html) 关于该主题的网页。

## 清理

如果你想删除本演示使用的所有资源，请运行此单元格（_警告：这将删除这些表以及其中插入的数据！_）：

In [32]:
session.execute(f"DROP TABLE IF EXISTS {keyspace}.philosophers_cql;")
session.execute(f"DROP TABLE IF EXISTS {keyspace}.philosophers_cql_partitioned;")